# 00 · Data sampling + tokenizer (Phase 1)
Run this once to prepare the tokenizer and training files.

It streams the four raw JSON files (19 GB total) and takes a document-level **10% sample** of each source,
holding out 0.5% of the sampled docs for validation. It then trains a **32k byte-level BPE**, tokenises everything
into `uint16` token files, and builds a local 512→128-word eval set. Everything is saved to Drive and reused
by the training notebooks.

In [ ]:
# ---- paths + locate the code (lm_lib/ and models/) ----
# Put the code on Drive in ONE of these ways:
#   (a) upload the folders lm_lib/ and models/ (+ hf_token.txt) into MyDrive/cs781_capstone/   [or any folder]
#   (b) upload cs781_code.zip (made by make_code_zip.py) anywhere in MyDrive -> it is unzipped automatically
from google.colab import drive
drive.mount('/content/drive')
import sys, os, glob, zipfile, importlib, shutil, time
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; must precede CUDA init
MYDRIVE    = '/content/drive/MyDrive'
DRIVE_ROOT = f'{MYDRIVE}/cs781'            # data + runs live here
DRIVE_DATA = f'{DRIVE_ROOT}/data'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
LOCAL_DATA = '/content/data'

def find_code_dir():
    levels = ['', '*/', '*/*/', '*/*/*/']
    # 1) a folder on Drive containing lm_lib/ (newest wins if there are several copies)
    hits = [h for lv in levels for h in glob.glob(f'{MYDRIVE}/{lv}lm_lib/trainer.py')]
    if hits:
        return os.path.dirname(os.path.dirname(max(hits, key=os.path.getmtime)))
    # 2) the newest cs781_code*.zip on Drive -> always re-extracted fresh (so an updated zip replaces old code)
    zips = [z for lv in levels for z in glob.glob(f'{MYDRIVE}/{lv}cs781_code*.zip')]
    if zips:
        z = max(zips, key=os.path.getmtime)
        shutil.rmtree('/content/cs781_code', ignore_errors=True)
        with zipfile.ZipFile(z) as f:
            f.extractall('/content/cs781_code')
        print('unzipped', z, '| zip modified', time.ctime(os.path.getmtime(z)))
        return '/content/cs781_code'
    raise FileNotFoundError('Could not find lm_lib/ on Drive. Upload the lm_lib/ and models/ folders '
                            '(e.g. to MyDrive/cs781_capstone/) or upload cs781_code.zip to MyDrive, then re-run.')

CODE_DIR = find_code_dir()
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
importlib.invalidate_caches()
# if lm_lib/models were imported earlier in this kernel, drop them so the fresh code is used
for _m in [m for m in list(sys.modules) if m == 'lm_lib' or m.startswith(('lm_lib.', 'models.')) or m == 'models']:
    del sys.modules[_m]
print('CODE_DIR =', CODE_DIR, '| contents:', sorted(os.listdir(CODE_DIR)))
os.makedirs(DRIVE_DATA, exist_ok=True); os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
!pip -q install ijson "huggingface_hub[hf_xet]" tokenizers
import os
from huggingface_hub import login, whoami

def get_hf_token():
    if os.environ.get('HF_TOKEN'):
        return os.environ['HF_TOKEN'].strip()
    p = f'{CODE_DIR}/hf_token.txt'
    if os.path.exists(p):
        return open(p).read().strip()
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception as e:
        raise RuntimeError(f'No HF token found: set $HF_TOKEN or create {p}') from e

HF_TOKEN = get_hf_token()
login(token=HF_TOKEN, add_to_git_credential=False)
print('logged in to Hugging Face as', whoami(token=HF_TOKEN)['name'])

In [ ]:
from huggingface_hub import hf_hub_download
REPO = 'Exploration-Lab/CS-781-Capstone'
RAW = '/content/raw'
SOURCES = {'acts': 'train_acts.json', 'high_courts': 'train_high_courts.json',
           'supreme_court': 'train_supreme_court.json', 'tribunals': 'train_tribunals.json'}
files = {}
for name, fn in SOURCES.items():
    files[name] = hf_hub_download(REPO, f'Phase_1_Pretraining/Train/{fn}', repo_type='dataset', local_dir=RAW,
                                  token=HF_TOKEN)
    print(name, files[name])
test_path = hf_hub_download(REPO, 'Phase_1_Pretraining/Test/test.json', repo_type='dataset', local_dir=RAW,
                            token=HF_TOKEN)
!du -sh {RAW}/Phase_1_Pretraining/*/*

In [ ]:
# look at the raw structure before trusting the parser
from lm_lib import data_prep as dp
for p in list(files.values()) + [test_path]:
    print(dp._detect_format(p)); dp.peek(p, 800)

### Sampling / mixing
`FRAC` can be a float (the same fraction for every source) or a per-source dict if you want to change the mix
(e.g. up-weight Supreme Court). Keep the seed fixed so the sample is reproducible for the report.

In [ ]:
FRAC = 0.10            # or {'acts': 0.10, 'high_courts': 0.10, 'supreme_court': 0.10, 'tribunals': 0.10}
VAL_FRAC = 0.005
NORMALIZE_WS = False   # True => collapse newlines/paragraph breaks to single spaces, like the official test prompts
WORK = '/content/work'
stats = dp.sample_sources(files, WORK, frac=FRAC, val_frac=VAL_FRAC, seed=1337, normalize_whitespace=NORMALIZE_WS)
!ls -lh {WORK}

In [ ]:
VOCAB = 32000
tok = dp.train_tokenizer(f'{WORK}/train_docs.jsonl', f'{WORK}/tokenizer.json', vocab_size=VOCAB,
                         max_chars=400_000_000, split_digits=False)

In [ ]:
tr = dp.tokenize_to_bin(f'{WORK}/train_docs.jsonl', f'{WORK}/train.bin', f'{WORK}/tokenizer.json')
va = dp.tokenize_to_bin(f'{WORK}/val_docs.jsonl', f'{WORK}/val.bin', f'{WORK}/tokenizer.json')
dp.make_eval_prompts(f'{WORK}/val_docs.jsonl', WORK, n=300)

In [ ]:
# how long are the official prompts in tokens? (decides seq_len=1024 / src_len=768)
import json, numpy as np
from tokenizers import Tokenizer
t = Tokenizer.from_file(f'{WORK}/tokenizer.json')
test = json.load(open(test_path))
lens = np.array([len(e.ids) for e in t.encode_batch([v['text'] for v in test.values()])])
print(f'{len(test)} test prompts; tokens per prompt: mean {lens.mean():.0f}, p50 {np.percentile(lens,50):.0f}, '
      f'p95 {np.percentile(lens,95):.0f}, max {lens.max()}')
refs = json.load(open(f'{WORK}/val_refs.json'))
cl = np.array([len(e.ids) for e in t.encode_batch([v['text'] for v in refs.values()])])
print(f'128-word continuations: mean {cl.mean():.0f} tokens, p95 {np.percentile(cl,95):.0f}')
print(t.decode(np.fromfile(f'{WORK}/train.bin', dtype=np.uint16, count=300).tolist())[:1000])

In [ ]:
import shutil
meta = dict(sample=stats, train=tr, val=va, vocab=VOCAB, frac=FRAC, val_frac=VAL_FRAC, normalize_ws=NORMALIZE_WS)
json.dump(meta, open(f'{WORK}/data_meta.json', 'w'), indent=2)
for f in ['train.bin', 'val.bin', 'tokenizer.json', 'val_prompts.json', 'val_refs.json', 'val_srcs.json',
          'sample_stats.json', 'data_meta.json']:
    shutil.copyfile(f'{WORK}/{f}', f'{DRIVE_DATA}/{f}')
shutil.copyfile(test_path, f'{DRIVE_DATA}/test.json')
!ls -lh {DRIVE_DATA}